# ZEDCREST CAPITAL - Data Analyst Training Programme
## Session 2 · Monday, May 11, 2026 · 7:00 PM - 9:00 PM WAT

---

> **Trainer:** Kajola Gbenga Adewale | Prodigy Training Hub  
> **Trainee:** Data Analysts, Zedcrest Capital  
> **Session Topic:** Control Flow & Functions in Python  
> **Applied Context:** Flagging NAV Anomalies & Automating Fund Checks  

---

## Session Agenda

| # | Topic | Time |
|---|-------|------|
| 1 | Recap of Session 1 + Dataset Introduction | 10 mins |
| 2 | Comparison Operators & Boolean Logic | 15 mins |
| 3 | `if / elif / else` Statements | 20 mins |
| 4 | Applying Conditions to Zedcrest NAV Data | 15 mins |
| 5 | `for` Loops - Iterating Over Fund Data | 15 mins |
| 6 | `while` Loops - Running Checks Until a Condition is Met | 10 mins |
| 7 | Defining Functions - Reusable Analytics Tools | 20 mins |
| 8 | Capstone Mini-Project: NAV Anomaly Flagging Script | 15 mins |

---

## Learning Objectives

By the end of this session, you will be able to:

- Use **comparison operators** and **boolean logic** to evaluate financial conditions
- Write **`if / elif / else`** statements to route decisions based on data
- Use **`for` loops** to process every row in a fund dataset automatically
- Use **`while` loops** to run monitoring checks until a threshold is breached
- Define **reusable functions** that encapsulate business logic (e.g. anomaly detection)
- Combine all of the above to build a working **NAV Anomaly Flagging Script** on real-style Zedcrest fund data

---

## The Zedcrest Connection

> *Everything we build today has a direct use case at Zedcrest Capital.*
>
> NAV (Net Asset Value) is the price of one unit in a fund. It changes daily based on the performance of the assets inside that fund. Monitoring NAV movements manually - scrolling through Excel looking for suspicious jumps or drops — is slow, error-prone, and unscalable.
>
> **Today, we automate it.** By the end of this session, you will have a Python script that reads NAV data, applies business rules, and flags any anomaly automatically - the same logic that powers professional risk monitoring systems.


---
## Section 0 - Generating Our Zedcrest Training Dataset

Before we write any control flow, we need data to work with. We will generate a realistic dataset representing Zedcrest Capital's daily NAV records across 5 funds × 120 trading days = **600 rows**.

> **What is NAV?**  
> NAV = (Total Assets - Total Liabilities) ÷ Units Outstanding  
> It tells an investor the current price of one unit in a fund. If NAV was ₦100 yesterday and ₦102 today, the fund returned +2% overnight.

### Dataset columns:

| Column | Description | Example |
|--------|-------------|--------|
| `date` | Trading date | 2026-01-02 |
| `fund_name` | Name of the Zedcrest fund | Zedcrest Money Market Fund |
| `fund_code` | Short code | ZMMF |
| `nav` | Net Asset Value per unit (₦) | ₦102.45 |
| `daily_return_pct` | Day-over-day NAV change (%) | +0.85% |
| `aum_millions` | Assets Under Management (₦ million) | ₦4,500M |
| `units_outstanding` | Fund units in circulation | 43,947,000 |
| `benchmark_return_pct` | Fund's benchmark return that day | +0.60% |
| `fund_manager` | Portfolio manager | Adebayo Okafor |
| `is_anomaly` | Ground truth label (we planted it) | True/False |
| `anomaly_type` | LARGE DROP, LARGE SPIKE, or None | LARGE DROP |


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 - Import libraries and define our five Zedcrest funds
# We use only Python's built-in libraries - no pip install needed yet.
# ─────────────────────────────────────────────────────────────────────────────

import random
import math
from datetime import date, timedelta

# Set a seed for reproducibility - same 'random' numbers every run
# This is critical in finance: analysis must always be reproducible
random.seed(42)

# Define our five Zedcrest funds with their financial characteristics
FUNDS = [
    {
        "fund_name":         "Zedcrest Money Market Fund",
        "fund_code":         "ZMMF",
        "start_nav":         100.00,
        "mean_daily_return": 0.045,   # ~11% annual return - typical money market
        "volatility":        0.02,    # very low - money market funds are stable
        "start_aum":         4500,    # ₦4,500 million AUM
        "benchmark_mean":    0.040,
        "fund_manager":      "Adebayo Okafor"
    },
    {
        "fund_name":         "Zedcrest Equity Growth Fund",
        "fund_code":         "ZEGF",
        "start_nav":         250.00,
        "mean_daily_return": 0.08,    # ~20% annual - equities grow faster but riskier
        "volatility":        0.85,    # high volatility
        "start_aum":         2800,
        "benchmark_mean":    0.07,
        "fund_manager":      "Chisom Nwosu"
    },
    {
        "fund_name":         "Zedcrest Fixed Income Fund",
        "fund_code":         "ZFIF",
        "start_nav":         150.00,
        "mean_daily_return": 0.055,   # ~14% annual — bonds are medium risk/return
        "volatility":        0.15,
        "start_aum":         6200,
        "benchmark_mean":    0.050,
        "fund_manager":      "Kemi Adeleke"
    },
    {
        "fund_name":         "Zedcrest Dollar Fund",
        "fund_code":         "ZDLF",
        "start_nav":         85.50,   # USD-denominated fund
        "mean_daily_return": 0.035,
        "volatility":        0.30,
        "start_aum":         1200,
        "benchmark_mean":    0.030,
        "fund_manager":      "Emeka Eze"
    },
    {
        "fund_name":         "Zedcrest Balanced Fund",
        "fund_code":         "ZBAF",
        "start_nav":         180.00,  # mix of equity + fixed income
        "mean_daily_return": 0.065,
        "volatility":        0.45,
        "start_aum":         3100,
        "benchmark_mean":    0.060,
        "fund_manager":      "Fatima Suleiman"
    },
]

print(f'Defined {len(FUNDS)} Zedcrest funds:')
for f in FUNDS:
    print(f'  {f["fund_code"]} — {f["fund_name"]} | Manager: {f["fund_manager"]} | Start NAV: ₦{f["start_nav"]}')


Defined 5 Zedcrest funds:
  ZMMF — Zedcrest Money Market Fund | Manager: Adebayo Okafor | Start NAV: ₦100.0
  ZEGF — Zedcrest Equity Growth Fund | Manager: Chisom Nwosu | Start NAV: ₦250.0
  ZFIF — Zedcrest Fixed Income Fund | Manager: Kemi Adeleke | Start NAV: ₦150.0
  ZDLF — Zedcrest Dollar Fund | Manager: Emeka Eze | Start NAV: ₦85.5
  ZBAF — Zedcrest Balanced Fund | Manager: Fatima Suleiman | Start NAV: ₦180.0


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 - Generate 120 trading days (Mon–Fri only, skipping weekends)
# ─────────────────────────────────────────────────────────────────────────────

def generate_trading_days(start_date, num_days):
    """
    Returns a list of weekday dates (Mon-Fri) starting from start_date.
    We skip Saturday (weekday=5) and Sunday (weekday=6).
    This mirrors the NGX (Nigerian Exchange Group) trading calendar.

    Args:
        start_date (date): First possible trading day
        num_days   (int):  How many trading days to generate
    Returns:
        list[date]: List of trading dates
    """
    trading_days = []
    current = start_date
    while len(trading_days) < num_days:
        if current.weekday() < 5:   # 0=Mon, 1=Tue, 2=Wed, 3=Thu, 4=Fri
            trading_days.append(current)
        current += timedelta(days=1)
    return trading_days


start_date   = date(2026, 1, 2)              # First trading day of 2026
trading_days = generate_trading_days(start_date, 120)

print(f"Generated {len(trading_days)} trading days")
print(f"First day : {trading_days[0]}  ({trading_days[0].strftime('%A')})")
print(f"Last day  : {trading_days[-1]}  ({trading_days[-1].strftime('%A')})")


Generated 120 trading days
First day : 2026-01-02  (Friday)
Last day  : 2026-06-18  (Thursday)


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Simulate daily NAV movements for all 5 funds over 120 days
#
# NAV logic: new_nav = old_nav × (1 + daily_return / 100)
# We use a Gaussian random walk — realistic market behaviour.
# Every ~5th day, we deliberately inject an anomaly (large drop or spike).
# This creates the 'needle in a haystack' our script must find later.
# ─────────────────────────────────────────────────────────────────────────────

def gaussian_noise():
    """
    Box-Muller transform: generates a normally-distributed random number.
    Most days, returns cluster near the mean.
    Occasionally, large outliers occur — like real markets.
    """
    u1 = random.random()
    u2 = random.random()
    # Box-Muller: z = sqrt(-2 × ln(u1)) × cos(2π × u2)
    z = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
    return z


dataset = []   # all rows go here — will have 600 entries (5 funds × 120 days)

for fund in FUNDS:
    nav   = fund["start_nav"]                          # starting NAV
    aum   = fund["start_aum"]                          # starting AUM (₦ millions)
    units = (aum * 1_000_000) / nav                    # units = total value / price per unit

    for day in trading_days:

        # ── Randomly inject anomalies (~5% of days) ───────────────────────────
        chance = random.random()   # number between 0 and 1

        if chance < 0.025:         # 2.5% chance: large DROP
            daily_return = random.uniform(-3.5, -2.0)   # -2% to -3.5%
            is_anomaly   = True
            anomaly_type = "LARGE DROP"

        elif chance < 0.05:        # another 2.5% chance: large SPIKE
            daily_return = random.uniform(2.5, 4.5)     # +2.5% to +4.5%
            is_anomaly   = True
            anomaly_type = "LARGE SPIKE"

        else:                      # 95% normal day
            noise        = gaussian_noise()
            daily_return = fund["mean_daily_return"] + noise * fund["volatility"]
            daily_return = max(-2.0, min(2.0, daily_return))   # cap at ±2%
            is_anomaly   = False
            anomaly_type = "None"

        # ── Benchmark return (always normal) ──────────────────────────────────
        benchmark = fund["benchmark_mean"] + gaussian_noise() * fund["volatility"] * 0.5
        benchmark = round(benchmark, 4)

        # ── Update NAV and AUM ─────────────────────────────────────────────────
        new_nav      = round(nav * (1 + daily_return / 100), 4)
        flow         = random.uniform(-0.005, 0.008)    # daily investor in/outflow
        units        = units * (1 + flow)
        aum_millions = round((new_nav * units) / 1_000_000, 2)

        # ── Append row to dataset ──────────────────────────────────────────────
        dataset.append({
            "date":                 day.strftime("%Y-%m-%d"),
            "fund_name":            fund["fund_name"],
            "fund_code":            fund["fund_code"],
            "fund_manager":         fund["fund_manager"],
            "nav":                  round(new_nav, 4),
            "daily_return_pct":     round(daily_return, 4),
            "benchmark_return_pct": benchmark,
            "aum_millions":         aum_millions,
            "units_outstanding":    round(units),
            "is_anomaly":           is_anomaly,
            "anomaly_type":         anomaly_type
        })
        nav = new_nav   # carry NAV forward

print(f"Dataset ready: {len(dataset)} rows")
print(f"Anomalies planted: {sum(1 for r in dataset if r['is_anomaly'])}")
print()
# Preview first 3 rows
print("First 3 rows:")
print("-" * 80)
for row in dataset[:3]:
    for k, v in row.items():
        print(f"  {k:<25} : {v}")
    print()


Dataset ready: 600 rows
Anomalies planted: 28

First 3 rows:
--------------------------------------------------------------------------------
  date                      : 2026-01-02
  fund_name                 : Zedcrest Money Market Fund
  fund_code                 : ZMMF
  fund_manager              : Adebayo Okafor
  nav                       : 100.0365
  daily_return_pct          : 0.0365
  benchmark_return_pct      : 0.0385
  aum_millions              : 4518.74
  units_outstanding         : 45170869
  is_anomaly                : False
  anomaly_type              : None

  date                      : 2026-01-05
  fund_name                 : Zedcrest Money Market Fund
  fund_code                 : ZMMF
  fund_manager              : Adebayo Okafor
  nav                       : 100.0425
  daily_return_pct          : 0.006
  benchmark_return_pct      : 0.0452
  aum_millions              : 4526.1
  units_outstanding         : 45241770
  is_anomaly                : False
  anomaly_type  

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Dataset Summary Statistics
# Always understand your data before you analyse it.
# ─────────────────────────────────────────────────────────────────────────────

total_rows     = len(dataset)
total_anom     = sum(1 for r in dataset if r['is_anomaly'])
drops          = sum(1 for r in dataset if r['anomaly_type'] == 'LARGE DROP')
spikes         = sum(1 for r in dataset if r['anomaly_type'] == 'LARGE SPIKE')

print("=" * 55)
print("  ZEDCREST NAV DATASET — SUMMARY")
print("=" * 55)
print(f"  Total rows         : {total_rows}")
print(f"  Anomalies planted  : {total_anom}")
print(f"    Large Drops      : {drops}")
print(f"    Large Spikes     : {spikes}")
print(f"  Anomaly rate       : {total_anom / total_rows * 100:.1f}%")
print("=" * 55)
print()
print("Breakdown by fund:")
print("-" * 55)
for fund in FUNDS:
    fc   = fund["fund_code"]
    fn   = fund["fund_name"]
    n    = sum(1 for r in dataset if r['fund_code'] == fc and r['is_anomaly'])
    print(f"  {fc}  ({fn:<32})  {n} anomalies")


  ZEDCREST NAV DATASET — SUMMARY
  Total rows         : 600
  Anomalies planted  : 28
    Large Drops      : 14
    Large Spikes     : 14
  Anomaly rate       : 4.7%

Breakdown by fund:
-------------------------------------------------------
  ZMMF  (Zedcrest Money Market Fund      )  7 anomalies
  ZEGF  (Zedcrest Equity Growth Fund     )  4 anomalies
  ZFIF  (Zedcrest Fixed Income Fund      )  6 anomalies
  ZDLF  (Zedcrest Dollar Fund            )  5 anomalies
  ZBAF  (Zedcrest Balanced Fund          )  6 anomalies


---
## Section 1 - Comparison Operators & Boolean Logic

Before we can build any decision-making system, Python needs a way to **compare values**.  
These are called **comparison operators** — they always return `True` or `False`.

### Python's Six Comparison Operators

| Operator | Meaning | Zedcrest Example |
|----------|---------|------------------|
| `==` | Equals | `fund_code == 'ZMMF'` |
| `!=` | Not equal | `anomaly_type != 'None'` |
| `>` | Greater than | `daily_return > 2.0` |
| `<` | Less than | `daily_return < -2.0` |
| `>=` | Greater or equal | `aum_millions >= 5000` |
| `<=` | Less or equal | `nav <= 90.0` |

> **Key rule:** `=` **assigns** a value. `==` **compares** values.  
> Confusing these is one of the most common beginner errors.

### Boolean Operators: `and`, `or`, `not`

| Operator | Meaning | Example |
|----------|---------|--------|
| `and` | BOTH must be True | `ret < -2 and aum > 5000` |
| `or` | AT LEAST ONE must be True | `ret < -2 or ret > 2.5` |
| `not` | Reverses the result | `not (code == 'ZMMF')` |


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1A: Comparison operators on a single fund record
# ─────────────────────────────────────────────────────────────────────────────

sample = dataset[0]   # grab the first record

print("Sample record:")
print(f"  Fund   : {sample['fund_name']}")
print(f"  Date   : {sample['date']}")
print(f"  NAV    : ₦{sample['nav']}")
print(f"  Return : {sample['daily_return_pct']}%")
print(f"  AUM    : ₦{sample['aum_millions']}M")
print()

# Apply each comparison operator
print("Comparison results:")
print("-" * 55)

# (1) ==  Is this the Money Market Fund?
result1 = (sample['fund_code'] == 'ZMMF')
print(f"  fund_code == 'ZMMF'     → {result1}")

# (2) !=  Is there an anomaly today?
result2 = (sample['anomaly_type'] != 'None')
print(f"  anomaly_type != 'None'  → {result2}")

# (3) >   Did the fund gain more than 1%?
result3 = (sample['daily_return_pct'] > 1.0)
print(f"  daily_return > 1.0%    → {result3}")

# (4) <   Did the fund drop more than 1%?
result4 = (sample['daily_return_pct'] < -1.0)
print(f"  daily_return < -1.0%   → {result4}")

# (5) >=  Is AUM at least ₦4,000M?
result5 = (sample['aum_millions'] >= 4000)
print(f"  aum_millions >= 4000M  → {result5}")

# (6) <=  Is NAV below or at ₦105?
result6 = (sample['nav'] <= 105.0)
print(f"  nav <= 105             → {result6}")


Sample record:
  Fund   : Zedcrest Money Market Fund
  Date   : 2026-01-02
  NAV    : ₦100.0365
  Return : 0.0365%
  AUM    : ₦4518.74M

Comparison results:
-------------------------------------------------------
  fund_code == 'ZMMF'     → True
  anomaly_type != 'None'  → False
  daily_return > 1.0%    → False
  daily_return < -1.0%   → False
  aum_millions >= 4000M  → True
  nav <= 105             → True


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1B: Boolean operators — combining conditions
# ─────────────────────────────────────────────────────────────────────────────

ret   = sample['daily_return_pct']
aum   = sample['aum_millions']
fc    = sample['fund_code']

print(f"Working with: {sample['fund_name']} | Return: {ret}% | AUM: ₦{aum}M")
print()

# AND: both conditions must be True
# Use case: alert only for large drops in large funds (tighter rules for bigger AUM)
big_drop_large_fund = (ret < -1.5) and (aum >= 3000)
print(f"  ret < -1.5% AND aum >= ₦3,000M  → {big_drop_large_fund}")

# OR: at least one condition must be True
# Use case: flag for review if the return is extreme in either direction
needs_review = (ret < -2.0) or (ret > 2.5)
print(f"  ret < -2.0% OR ret > 2.5%       → {needs_review}")

# NOT: reverse the result
# Use case: process all non-money-market funds (they have different rules)
not_mm = not (fc == 'ZMMF')
print(f"  NOT (fund_code == 'ZMMF')        → {not_mm}")

print()
print("Truth Table — How and/or behave:")
print("-" * 40)
print(f"  True  and True   → {True and True}")
print(f"  True  and False  → {True and False}")
print(f"  False and False  → {False and False}")
print(f"  True  or  False  → {True or False}")
print(f"  False or  False  → {False or False}")
print(f"  not True         → {not True}")
print(f"  not False        → {not False}")


Working with: Zedcrest Money Market Fund | Return: 0.0365% | AUM: ₦4518.74M

  ret < -1.5% AND aum >= ₦3,000M  → False
  ret < -2.0% OR ret > 2.5%       → False
  NOT (fund_code == 'ZMMF')        → False

Truth Table — How and/or behave:
----------------------------------------
  True  and True   → True
  True  and False  → False
  False and False  → False
  True  or  False  → True
  False or  False  → False
  not True         → False
  not False        → True


---
## Section 2 — `if / elif / else` — Making Decisions with Data

Now that we can compare values, we can make **decisions**.

The `if / elif / else` structure routes code execution based on conditions.

```python
if condition_1:
    # runs if condition_1 is True
elif condition_2:
    # runs if condition_1 False AND condition_2 True
else:
    # runs if ALL conditions above are False
```

### Indentation is Non-Negotiable
Python uses **4 spaces** (or 1 tab) to define code blocks. Forgetting indentation raises an `IndentationError`.

### Zedcrest NAV Return Classification Rules

| Daily Return | Category | Action |
|-------------|----------|--------|
| < −2.0% | 🔴 CRITICAL ALERT | Notify MD immediately |
| −2.0% to −1.0% | 🟠 WARNING | Risk committee review |
| −1.0% to 0.0% | 🟡 MILD NEGATIVE | Monitor |
| 0.0% to +1.5% | 🟢 NORMAL | No action |
| +1.5% to +2.5% | 🔵 STRONG | Log for reporting |
| > +2.5% | ⚠️ SPIKE ALERT | Verify data integrity |


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2A: Basic if / elif / else
# Classify a single fund return — start simple, then build complexity.
# ─────────────────────────────────────────────────────────────────────────────

daily_return = -2.35   # test value: fund dropped 2.35% today

print(f"Daily Return: {daily_return}%")
print()

# Python checks conditions TOP TO BOTTOM, stopping at the first True match
if daily_return < -2.0:
    # Only runs if return is worse than -2%
    category = "🔴 CRITICAL ALERT"
    action   = "Notify MD and Risk Manager IMMEDIATELY. Verify with custodian."

elif daily_return < -1.0:
    # Runs if first condition was False AND this is True (i.e. between -2% and -1%)
    category = "🟠 WARNING"
    action   = "Flag for risk committee review. Monitor closely."

elif daily_return < 0.0:
    # Between -1.0% and 0.0%
    category = "🟡 MILD NEGATIVE"
    action   = "Log and monitor. No immediate action required."

elif daily_return <= 1.5:
    # Between 0.0% and +1.5%
    category = "🟢 NORMAL"
    action   = "No action needed."

elif daily_return <= 2.5:
    # Between +1.5% and +2.5%
    category = "🔵 STRONG PERFORMANCE"
    action   = "Log for monthly performance report."

else:
    # Above +2.5% — could be a data entry error
    category = "⚠️ SPIKE ALERT"
    action   = "Verify with fund administrator. Possible data error."

print(f"Category : {category}")
print(f"Action   : {action}")

# Now try changing daily_return to these values and observe the output:
# daily_return = 0.45   → NORMAL
# daily_return = -1.5   → WARNING
# daily_return = 3.2    → SPIKE ALERT


Daily Return: -2.35%

Category : 🔴 CRITICAL ALERT
Action   : Notify MD and Risk Manager IMMEDIATELY. Verify with custodian.


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2B: Nested if statements — multi-factor decision making
#
# Real Zedcrest risk alerts depend on MULTIPLE factors:
#   1. Which fund is it? (Money Market has tighter thresholds)
#   2. How severe is the return?
#   3. How large is the fund? (larger funds = stricter monitoring)
# ─────────────────────────────────────────────────────────────────────────────

def classify_nav_alert(fund_code, daily_return, aum_millions):
    """
    Classify a NAV movement into an alert level.
    Thresholds differ by fund type — money market funds must be much more stable.

    Args:
        fund_code    (str):   e.g. 'ZMMF', 'ZEGF'
        daily_return (float): daily return percentage, e.g. -2.35
        aum_millions (float): AUM in naira millions, e.g. 4500.0
    Returns:
        tuple: (alert_level str, message str)
    """

    if fund_code == 'ZMMF':
        # Money Market Fund: extremely stable by design.
        # Even -0.5% is alarming. Nested if handles the sub-cases.
        if daily_return < -0.5:
            return ("🔴 CRITICAL", f"ZMMF dropped {daily_return:.2f}% — extremely unusual!")
        elif daily_return < 0.0:
            return ("🟠 WARNING",  f"ZMMF returned {daily_return:.2f}% — negative MM return.")
        elif daily_return > 1.0:
            return ("⚠️ SPIKE",   f"ZMMF spiked {daily_return:.2f}% — verify data.")
        else:
            return ("🟢 NORMAL",  f"ZMMF returned {daily_return:.2f}% — within expected range.")

    else:
        # All other funds: wider thresholds (equities/bonds move more)
        if daily_return < -2.0:
            alert = "🔴 CRITICAL"
            msg   = f"{fund_code} dropped {daily_return:.2f}% in one day."
            # Nested: escalate further if fund is very large
            if aum_millions >= 5000:
                msg += " ⚡ HIGH PRIORITY — AUM > ₦5B. Escalate to Board Risk Committee."
            else:
                msg += " Notify MD and Risk Manager."

        elif daily_return < -1.0:
            alert = "🟠 WARNING"
            msg   = f"{fund_code} returned {daily_return:.2f}%. Flag for risk review."

        elif daily_return > 2.5:
            alert = "⚠️ SPIKE"
            msg   = f"{fund_code} spiked {daily_return:.2f}%. Verify with fund admin."

        elif daily_return > 0.0:
            alert = "🟢 NORMAL"
            msg   = f"{fund_code} returned +{daily_return:.2f}%. All clear."

        else:
            alert = "🟡 MILD NEG"
            msg   = f"{fund_code} returned {daily_return:.2f}%. Monitor."

        return (alert, msg)


# Test across all fund types and scenarios
test_cases = [
    ('ZMMF', -0.80, 4500),   # Money market — large drop
    ('ZMMF',  0.04, 4500),   # Money market — normal day
    ('ZEGF', -3.20, 2800),   # Equity — crash
    ('ZFIF',  0.06, 6200),   # Fixed income — normal
    ('ZDLF',  3.10, 1200),   # Dollar fund — spike
    ('ZBAF', -1.50, 3100),   # Balanced — warning
    ('ZFIF', -2.50, 6200),   # Fixed income LARGE fund — critical + escalate
]

print("=" * 70)
print("  NAV CLASSIFICATION — Multi-Scenario Test")
print("=" * 70)
for fc, ret, aum in test_cases:
    level, msg = classify_nav_alert(fc, ret, aum)
    print(f"  [{level:<14}] {msg}")


  NAV CLASSIFICATION — Multi-Scenario Test
  [🔴 CRITICAL    ] ZMMF dropped -0.80% — extremely unusual!
  [🟢 NORMAL      ] ZMMF returned 0.04% — within expected range.
  [🔴 CRITICAL    ] ZEGF dropped -3.20% in one day. Notify MD and Risk Manager.
  [🟢 NORMAL      ] ZFIF returned +0.06%. All clear.
  [⚠️ SPIKE      ] ZDLF spiked 3.10%. Verify with fund admin.
  [🟠 WARNING     ] ZBAF returned -1.50%. Flag for risk review.
  [🔴 CRITICAL    ] ZFIF dropped -2.50% in one day. ⚡ HIGH PRIORITY — AUM > ₦5B. Escalate to Board Risk Committee.


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2C: Ternary operator (one-line if)
#
# Syntax:  value_if_true  if  condition  else  value_if_false
# Excel equivalent:  =IF(condition, value_if_true, value_if_false)
# ─────────────────────────────────────────────────────────────────────────────

sample_returns = [0.85, -1.20, 0.03, -2.45, 0.60, -0.12]

print("Ternary operator — one-line if/else:")
print("-" * 45)
for r in sample_returns:
    # Read as: 'POSITIVE if r >= 0, otherwise NEGATIVE'
    label = 'POSITIVE 📈' if r >= 0 else 'NEGATIVE 📉'
    print(f"  Return: {r:+.2f}%  →  {label}")

print()
print("Real-world use — flag vs no flag in one line:")
print("-" * 45)
for r in sample_returns:
    flag = '⚠️  FLAGGED' if abs(r) > 1.5 else '✅ Normal'
    print(f"  Return: {r:+.2f}%  →  {flag}")


Ternary operator — one-line if/else:
---------------------------------------------
  Return: +0.85%  →  POSITIVE 📈
  Return: -1.20%  →  NEGATIVE 📉
  Return: +0.03%  →  POSITIVE 📈
  Return: -2.45%  →  NEGATIVE 📉
  Return: +0.60%  →  POSITIVE 📈
  Return: -0.12%  →  NEGATIVE 📉

Real-world use — flag vs no flag in one line:
---------------------------------------------
  Return: +0.85%  →  ✅ Normal
  Return: -1.20%  →  ✅ Normal
  Return: +0.03%  →  ✅ Normal
  Return: -2.45%  →  ⚠️  FLAGGED
  Return: +0.60%  →  ✅ Normal
  Return: -0.12%  →  ✅ Normal


---
## 🔁 Section 3 — `for` Loops — Processing Every Fund Record Automatically

The moment you have more than one row, you need a loop.

A `for` loop says: *'For each item in this collection, do the following.'*

**Without a loop:** Copy-paste the same 10 lines of code 600 times.  
**With a loop:** Write the code once. It runs 600 times automatically.

```python
for variable in collection:
    # variable takes the value of each item in turn
    # code runs once per item
```

> **Excel equivalent:** Like dragging a formula down an entire column — but programmable.


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3A: Basic for loop over a list
# ─────────────────────────────────────────────────────────────────────────────

fund_names = [
    "Zedcrest Money Market Fund",
    "Zedcrest Equity Growth Fund",
    "Zedcrest Fixed Income Fund",
    "Zedcrest Dollar Fund",
    "Zedcrest Balanced Fund"
]

print("Zedcrest Fund Roster:")
print("-" * 42)
# The loop variable 'fund' takes each value in fund_names, one at a time
for fund in fund_names:
    print(f"  → {fund}")


Zedcrest Fund Roster:
------------------------------------------
  → Zedcrest Money Market Fund
  → Zedcrest Equity Growth Fund
  → Zedcrest Fixed Income Fund
  → Zedcrest Dollar Fund
  → Zedcrest Balanced Fund


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3B: enumerate() — loop with a built-in counter
#
# enumerate() gives you BOTH the index (position number) AND the value.
# Use it whenever you need to know 'which row am I processing right now?'
# ─────────────────────────────────────────────────────────────────────────────

print("Zedcrest Fund Roster (numbered):")
print("-" * 42)

# start=1 makes our count begin at 1 instead of 0
for i, fund in enumerate(fund_names, start=1):
    print(f"  Fund #{i}: {fund}")


Zedcrest Fund Roster (numbered):
------------------------------------------
  Fund #1: Zedcrest Money Market Fund
  Fund #2: Zedcrest Equity Growth Fund
  Fund #3: Zedcrest Fixed Income Fund
  Fund #4: Zedcrest Dollar Fund
  Fund #5: Zedcrest Balanced Fund


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3C: Loop over the dataset — extract records for one fund
#
# Our dataset is a list of dictionaries.
# Each dictionary = one fund-day record.
# This pattern — loop + filter — is the foundation of data processing.
# ─────────────────────────────────────────────────────────────────────────────

# Extract all records for ZMMF (Money Market Fund)
zmmf_records = []                   # start with empty list

for row in dataset:                 # loop over all 600 rows
    if row['fund_code'] == 'ZMMF':  # check if this row belongs to ZMMF
        zmmf_records.append(row)    # if yes, add it to our list

print(f"Total ZMMF records extracted: {len(zmmf_records)}")
print()
print("First 5 ZMMF records:")
print("-" * 75)
for i, r in enumerate(zmmf_records[:5], start=1):
    print(
        f"  Day {i}: {r['date']}  "
        f"NAV=₦{r['nav']:.4f}  "
        f"Return={r['daily_return_pct']:+.4f}%  "
        f"AUM=₦{r['aum_millions']:.1f}M"
    )


Total ZMMF records extracted: 120

First 5 ZMMF records:
---------------------------------------------------------------------------
  Day 1: 2026-01-02  NAV=₦100.0365  Return=+0.0365%  AUM=₦4518.7M
  Day 2: 2026-01-05  NAV=₦100.0425  Return=+0.0060%  AUM=₦4526.1M
  Day 3: 2026-01-06  NAV=₦102.9414  Return=+2.8977%  AUM=₦4647.3M
  Day 4: 2026-01-07  NAV=₦103.0011  Return=+0.0580%  AUM=₦4647.3M
  Day 5: 2026-01-08  NAV=₦103.0443  Return=+0.0419%  AUM=₦4677.2M


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3D: Running statistics with a loop
#
# For each fund: total return, positive/negative days, best/worst day.
# This is the monthly performance summary a Zedcrest analyst would produce.
# ─────────────────────────────────────────────────────────────────────────────

fund_codes = ['ZMMF', 'ZEGF', 'ZFIF', 'ZDLF', 'ZBAF']

print("=" * 78)
print(
    f"  {'Fund':<6}  {'Pos Days':>8}  {'Neg Days':>8}"
    f"  {'Best Day':>10}  {'Worst Day':>10}  {'Total Ret':>10}"
)
print("=" * 78)

for fc in fund_codes:
    # Extract records for this fund using a list comprehension
    # (A compressed for loop — we'll cover comprehensions in Session 3)
    fund_rows = [r for r in dataset if r['fund_code'] == fc]

    positive_days = 0       # days where return > 0
    negative_days = 0       # days where return <= 0
    best_day      = -999    # track highest single-day return
    worst_day     = 999     # track lowest single-day return
    cumulative    = 1.0     # compound return tracker (starts at 1.0 = 100%)

    for row in fund_rows:
        ret = row['daily_return_pct']

        # Count positive and negative days
        if ret > 0:
            positive_days += 1
        else:
            negative_days += 1

        # Track extreme days
        if ret > best_day:  best_day  = ret
        if ret < worst_day: worst_day = ret

        # Compound return: multiply factors together
        # e.g. +1% means × 1.0100, -2% means × 0.9800
        cumulative *= (1 + ret / 100)

    total_return = (cumulative - 1) * 100   # convert back to %

    print(
        f"  {fc:<6}  {positive_days:>8}  {negative_days:>8}"
        f"  {best_day:>+9.2f}%  {worst_day:>+9.2f}%  {total_return:>+9.2f}%"
    )

print("=" * 78)


  Fund    Pos Days  Neg Days    Best Day   Worst Day   Total Ret
  ZMMF         115         5      +3.79%      -3.39%      +3.32%
  ZEGF          62        58      +3.17%      -3.05%     +11.60%
  ZFIF          61        59      +4.37%      -3.03%      +6.43%
  ZDLF          59        61      +4.36%      -3.40%     +10.18%
  ZBAF          67        53      +4.38%      -3.02%      +6.49%


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3E: break and continue
#
# break    → STOP the loop immediately, exit it entirely
# continue → SKIP the rest of THIS iteration, go straight to the next one
# ─────────────────────────────────────────────────────────────────────────────

# ── break: Find the FIRST critical anomaly and stop scanning ─────────────────
# In a live monitoring system, you want to catch the first critical event
# immediately, not wait for the full scan to complete.

print("Scanning for first CRITICAL anomaly (return < -2%)...")
print()

first_critical = None

for row in dataset:
    if row['daily_return_pct'] < -2.0:
        first_critical = row
        break   # STOP — we found what we need. No point continuing the scan.

if first_critical:
    print(f"  ⚠️  First critical event found:")
    print(f"     Fund   : {first_critical['fund_name']}")
    print(f"     Date   : {first_critical['date']}")
    print(f"     Return : {first_critical['daily_return_pct']:+.2f}%")
    print(f"     NAV    : ₦{first_critical['nav']:.4f}")
    print(f"     AUM    : ₦{first_critical['aum_millions']:.1f}M")
else:
    print("  ✅  No critical anomalies found.")

print()

# ── continue: Skip rows we don't need, process only what matters ──────────────
# Compute average return for ZEGF, skipping any zero-return day (data errors)

print("Average daily return for ZEGF (excluding zero-return days):")

total = 0.0
count = 0

for row in dataset:
    if row['fund_code'] != 'ZEGF':
        continue   # skip — not ZEGF
    if row['daily_return_pct'] == 0.0:
        continue   # skip — suspicious zero return

    total += row['daily_return_pct']
    count += 1

avg = total / count if count > 0 else 0
print(f"  ZEGF average return: {avg:+.4f}%  ({count} valid trading days)")


Scanning for first CRITICAL anomaly (return < -2%)...

  ⚠️  First critical event found:
     Fund   : Zedcrest Money Market Fund
     Date   : 2026-01-27
     Return : -2.42%
     NAV    : ₦104.0942
     AUM    : ₦4781.8M

Average daily return for ZEGF (excluding zero-return days):
  ZEGF average return: +0.0958%  (120 valid trading days)


---
## Section 4 — `while` Loops — Running Until a Condition Changes

A `while` loop keeps running **as long as** a condition remains True.  
Unlike `for` (which iterates over a fixed collection), `while` is open-ended.

```python
while condition:
    # runs as long as condition is True
    # IMPORTANT: something inside must eventually make condition False
    # or you get an infinite loop!
```

### ⚠️ The Infinite Loop Trap
If the condition never becomes `False`, your program runs forever.  
Always ensure something inside the loop changes the condition.

### Zedcrest Use Cases
- Monitor NAV day by day until it drops below a minimum threshold (stop-loss)
- Scan each fund's history until the first stress event is found
- Run a daily data quality check until a corrupted record is detected


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4A: Basic while loop — countdown to trading halt
# ─────────────────────────────────────────────────────────────────────────────

# Circuit breakers halt trading when a fund hits a limit-down threshold.
# Here we simulate a countdown before the halt is triggered.

print("Market Circuit Breaker Countdown:")
print("-" * 40)

countdown = 5   # start at 5 seconds

while countdown > 0:             # keep looping as long as countdown > 0
    print(f"  Trading halts in {countdown} seconds...")
    countdown -= 1               # CRUCIAL: decrement the counter every iteration
                                 # Without this, countdown never reaches 0 → infinite loop!

print("  🚨 TRADING HALTED — Circuit breaker triggered!")


Market Circuit Breaker Countdown:
----------------------------------------
  Trading halts in 5 seconds...
  Trading halts in 4 seconds...
  Trading halts in 3 seconds...
  Trading halts in 2 seconds...
  Trading halts in 1 seconds...
  🚨 TRADING HALTED — Circuit breaker triggered!


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4B: while loop for NAV threshold monitoring
#
# Use case: Walk through ZMMF's NAV history day by day.
# Stop the moment NAV drops below ₦99.50 ('breaking the buck').
#
# 'Breaking the buck' = a money market fund NAV falling below starting value.
# This is a crisis event — extremely rare and signals serious problems.
# ─────────────────────────────────────────────────────────────────────────────

NAV_FLOOR = 99.50   # alert threshold — below this is critical

zmmf_rows = [r for r in dataset if r['fund_code'] == 'ZMMF']

print(f"Monitoring ZMMF NAV — alert if NAV drops below ₦{NAV_FLOOR}")
print("-" * 60)

i = 0   # day index — we use this to walk through the records

# Loop continues while:
# (1) we have records left (i < len(zmmf_rows))
# (2) NAV is still above the floor
while i < len(zmmf_rows) and zmmf_rows[i]['nav'] >= NAV_FLOOR:
    if i % 20 == 0:   # print every 20th day to keep output clean
        r = zmmf_rows[i]
        print(f"  Day {i+1:>3}  |  {r['date']}  |  NAV: ₦{r['nav']:.4f}  ✅")
    i += 1   # move to next day — CRUCIAL to prevent infinite loop

# Check WHY the loop stopped
if i >= len(zmmf_rows):
    print()
    print("  ✅ 120 days complete — ZMMF never breached the floor.")
else:
    r = zmmf_rows[i]
    print()
    print(f"  🚨 THRESHOLD BREACHED on Day {i + 1}!")
    print(f"     Date   : {r['date']}")
    print(f"     NAV    : ₦{r['nav']:.4f}  (below ₦{NAV_FLOOR})")
    print(f"     Return : {r['daily_return_pct']:+.2f}%")
    print(f"     Action : Escalate to MD and Fund Trustees IMMEDIATELY.")


Monitoring ZMMF NAV — alert if NAV drops below ₦99.5
------------------------------------------------------------
  Day   1  |  2026-01-02  |  NAV: ₦100.0365  ✅
  Day  21  |  2026-01-30  |  NAV: ₦104.2174  ✅
  Day  41  |  2026-02-27  |  NAV: ₦105.1292  ✅
  Day  61  |  2026-03-27  |  NAV: ₦102.6366  ✅

  🚨 THRESHOLD BREACHED on Day 71!
     Date   : 2026-04-10
     NAV    : ₦97.6012  (below ₦99.5)
     Return : -3.13%
     Action : Escalate to MD and Fund Trustees IMMEDIATELY.


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4C: while + break — first stress event per fund
# ─────────────────────────────────────────────────────────────────────────────

fund_codes = ['ZMMF', 'ZEGF', 'ZFIF', 'ZDLF', 'ZBAF']

print("First Stress Event (return < -2%) Per Fund:")
print("=" * 65)

for fc in fund_codes:
    fund_rows = [r for r in dataset if r['fund_code'] == fc]

    i     = 0
    found = False

    while i < len(fund_rows):
        row = fund_rows[i]

        if row['daily_return_pct'] < -2.0:
            print(
                f"  {fc}  →  {row['date']}  "
                f"Return: {row['daily_return_pct']:+.2f}%  "
                f"NAV: ₦{row['nav']:.4f}"
            )
            found = True
            break   # stop scanning this fund — first stress event found

        i += 1

    if not found:
        print(f"  {fc}  →  No stress event in 120-day period  ✅")

print("=" * 65)


First Stress Event (return < -2%) Per Fund:
  ZMMF  →  2026-01-27  Return: -2.42%  NAV: ₦104.0942
  ZEGF  →  2026-04-20  Return: -3.05%  NAV: ₦276.1425
  ZFIF  →  2026-01-02  Return: -2.32%  NAV: ₦146.5242
  ZDLF  →  2026-04-29  Return: -3.40%  NAV: ₦95.7083
  ZBAF  →  2026-01-19  Return: -2.12%  NAV: ₦180.5017


---
## Section 5 — Defining Functions — Reusable Analytics Tools

Functions are the single most important tool for writing **professional, maintainable code**.

A function is a **named, reusable block of code** that:
1. Takes **inputs** (parameters)
2. Does some **processing**
3. Returns an **output**

```python
def function_name(parameter1, parameter2):
    """
    Docstring: always explain what the function does.
    Good docstrings make code self-documenting.
    """
    result = parameter1 + parameter2
    return result
```

### Why Functions Matter at Zedcrest

**Without functions:** Copy-paste the same 10 lines of NAV-checking logic across 5 scripts. If the business rules change, update it in 5 places — and you'll miss one.

**With functions:** Write the logic once. Call it everywhere. Update once.

| Keyword | Purpose |
|---------|--------|
| `def` | Defines the function |
| function name | What you'll call it (use lowercase_with_underscores) |
| parameters | Inputs the function receives |
| `return` | The output sent back to the caller |
| `"""docstring"""` | Documentation — always write one |

> **Excel equivalent:** A function is like a named custom formula. Except a Python function can do *anything*.


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5A: First function — calculate daily return
# Formula: ((NAV_today - NAV_yesterday) / NAV_yesterday) × 100
# This is one of the most fundamental calculations in asset management.
# ─────────────────────────────────────────────────────────────────────────────

def calculate_daily_return(nav_today, nav_yesterday):
    """
    Calculate the percentage return for a fund over one day.

    Formula: ((NAV_today - NAV_yesterday) / NAV_yesterday) × 100

    Args:
        nav_today     (float): Today's NAV per unit
        nav_yesterday (float): Yesterday's NAV per unit
    Returns:
        float: Daily return as a percentage (e.g. 1.25 means +1.25%)
    """
    if nav_yesterday == 0:
        return 0.0   # guard against division by zero

    daily_return = ((nav_today - nav_yesterday) / nav_yesterday) * 100
    return round(daily_return, 4)


# Test with known values
print("Testing calculate_daily_return():")
print("-" * 55)

test_navs = [
    (102.45, 100.00),   # should be +2.45%
    (99.10,  100.00),   # should be -0.90%
    (100.00, 100.00),   # should be 0.00%
    (175.25, 173.10),   # small positive
    (85.10,   88.00),   # negative
]

for today, yesterday in test_navs:
    result = calculate_daily_return(today, yesterday)
    print(f"  ₦{yesterday:.2f} → ₦{today:.2f}  |  Return: {result:+.4f}%")


Testing calculate_daily_return():
-------------------------------------------------------
  ₦100.00 → ₦102.45  |  Return: +2.4500%
  ₦100.00 → ₦99.10  |  Return: -0.9000%
  ₦100.00 → ₦100.00  |  Return: +0.0000%
  ₦173.10 → ₦175.25  |  Return: +1.2421%
  ₦88.00 → ₦85.10  |  Return: -3.2955%


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5B: Functions with default parameters
#
# Some parameters have sensible default values.
# Define them with: parameter=default_value
# Callers can override, or just let the default apply.
# ─────────────────────────────────────────────────────────────────────────────

def classify_return(daily_return_pct, fund_type='EQUITY'):
    """
    Classify a daily return as NORMAL, WARNING, or CRITICAL.
    Thresholds differ by fund type: money market funds are held to tighter standards.

    Args:
        daily_return_pct (float): Daily return in percent
        fund_type        (str):   'MONEY_MARKET' or 'EQUITY' (default: 'EQUITY')
    Returns:
        dict: {'level': str, 'emoji': str, 'action': str}
    """
    if fund_type == 'MONEY_MARKET':
        # Tight thresholds: money markets must barely move
        if daily_return_pct < -0.5:
            return {'level': 'CRITICAL', 'emoji': '🔴', 'action': 'Notify MD immediately'}
        elif daily_return_pct < 0.0:
            return {'level': 'WARNING',  'emoji': '🟠', 'action': 'Risk committee review'}
        elif daily_return_pct > 1.0:
            return {'level': 'SPIKE',    'emoji': '⚠️',  'action': 'Verify with administrator'}
        else:
            return {'level': 'NORMAL',   'emoji': '🟢', 'action': 'No action required'}
    else:   # EQUITY and all other funds
        if daily_return_pct < -2.0:
            return {'level': 'CRITICAL', 'emoji': '🔴', 'action': 'Notify MD immediately'}
        elif daily_return_pct < -1.0:
            return {'level': 'WARNING',  'emoji': '🟠', 'action': 'Risk committee review'}
        elif daily_return_pct > 2.5:
            return {'level': 'SPIKE',    'emoji': '⚠️',  'action': 'Verify with administrator'}
        else:
            return {'level': 'NORMAL',   'emoji': '🟢', 'action': 'No action required'}


# Same -0.60% return — different outcome depending on fund type
ret = -0.60
print(f"Return: {ret:+.2f}%")
print("-" * 60)

# Using default (EQUITY)
r1 = classify_return(ret)
print(f"  Equity Fund       → {r1['emoji']} {r1['level']:<10} | {r1['action']}")

# Overriding the default
r2 = classify_return(ret, fund_type='MONEY_MARKET')
print(f"  Money Market Fund → {r2['emoji']} {r2['level']:<10} | {r2['action']}")

print()
print("Lesson: The SAME return triggers different responses depending on fund type.")


Return: -0.60%
------------------------------------------------------------
  Equity Fund       → 🟢 NORMAL     | No action required
  Money Market Fund → 🔴 CRITICAL   | Notify MD immediately

Lesson: The SAME return triggers different responses depending on fund type.


In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5C: Functions calling other functions
#
# Professional code has layers:
#   - Small helper functions each do ONE thing well.
#   - A higher-level function orchestrates them.
# This is called 'separation of concerns' — a key software design principle.
# ─────────────────────────────────────────────────────────────────────────────

def is_anomalous(daily_return_pct, threshold=2.0):
    """
    Return True if the absolute return exceeds the threshold.

    Args:
        daily_return_pct (float): Return in percent
        threshold        (float): Anomaly threshold (default 2.0%)
    Returns:
        bool
    """
    return abs(daily_return_pct) > threshold


def get_anomaly_direction(daily_return_pct):
    """
    Classify whether an anomaly is a DROP or SPIKE.

    Args:
        daily_return_pct (float): The flagged return
    Returns:
        str: 'LARGE DROP', 'LARGE SPIKE', or 'NORMAL'
    """
    if daily_return_pct < 0:
        return 'LARGE DROP'
    elif daily_return_pct > 0:
        return 'LARGE SPIKE'
    return 'NORMAL'


def analyse_record(record, threshold=2.0):
    """
    Fully analyse a single fund-day record.
    Calls is_anomalous() and get_anomaly_direction() internally.

    Args:
        record    (dict):  One row from the dataset
        threshold (float): Detection threshold (default 2.0%)
    Returns:
        dict: Full analysis result
    """
    ret      = record['daily_return_pct']
    flagged  = is_anomalous(ret, threshold)         # call helper 1
    anom_dir = get_anomaly_direction(ret) if flagged else 'NONE'  # call helper 2

    if not flagged:
        action = '✅ No action'
    elif ret < 0:
        action = '🔴 Contact fund administrator and custodian. Notify MD.'
    else:
        action = '⚠️  Verify with administrator — possible data entry error.'

    return {
        'date':    record['date'],
        'fund':    record['fund_code'],
        'return':  ret,
        'nav':     record['nav'],
        'flagged': flagged,
        'type':    anom_dir,
        'action':  action
    }


# Test on first 12 records
print("analyse_record() — Testing on first 12 dataset rows:")
print("=" * 80)
for record in dataset[:12]:
    r = analyse_record(record)
    marker = '  ⚠️ ' if r['flagged'] else '     '
    print(
        f"{marker}{r['fund']}  {r['date']}  "
        f"Return={r['return']:+.2f}%  NAV=₦{r['nav']:.2f}  "
        f"Flagged={str(r['flagged']):<5}  {r['type']}"
    )


analyse_record() — Testing on first 12 dataset rows:
     ZMMF  2026-01-02  Return=+0.04%  NAV=₦100.04  Flagged=False  NONE
     ZMMF  2026-01-05  Return=+0.01%  NAV=₦100.04  Flagged=False  NONE
  ⚠️ ZMMF  2026-01-06  Return=+2.90%  NAV=₦102.94  Flagged=True   LARGE SPIKE
     ZMMF  2026-01-07  Return=+0.06%  NAV=₦103.00  Flagged=False  NONE
     ZMMF  2026-01-08  Return=+0.04%  NAV=₦103.04  Flagged=False  NONE
     ZMMF  2026-01-09  Return=+0.04%  NAV=₦103.09  Flagged=False  NONE
     ZMMF  2026-01-12  Return=+0.04%  NAV=₦103.13  Flagged=False  NONE
  ⚠️ ZMMF  2026-01-13  Return=+2.96%  NAV=₦106.17  Flagged=True   LARGE SPIKE
     ZMMF  2026-01-14  Return=+0.02%  NAV=₦106.20  Flagged=False  NONE
     ZMMF  2026-01-15  Return=+0.04%  NAV=₦106.24  Flagged=False  NONE
     ZMMF  2026-01-16  Return=+0.07%  NAV=₦106.32  Flagged=False  NONE
     ZMMF  2026-01-19  Return=+0.05%  NAV=₦106.37  Flagged=False  NONE


---
## Section 6 — Capstone: The Zedcrest NAV Anomaly Flagging Script

This is the **session deliverable**. It combines everything we've learned today:

- ✅ Comparison operators and boolean logic
- ✅ `if / elif / else` classification
- ✅ `for` loops to process all 600 records
- ✅ Functions for reusable business logic
- ✅ Real (generated) Zedcrest data

### What this script does:
1. Scans all 600 fund-day records automatically
2. Applies configurable thresholds per fund type
3. Classifies each anomaly as DROP or SPIKE, with severity levels
4. Produces a full anomaly report grouped by fund
5. Evaluates its own accuracy against the ground truth (anomalies we planted)

> **This replaces a manual Excel process.**  
> A risk analyst scrolling through NAV data for 30 minutes each morning → replaced by this script running in milliseconds.


In [21]:
# ═════════════════════════════════════════════════════════════════════════════
# ZEDCREST CAPITAL — NAV ANOMALY FLAGGING SCRIPT
# Session 2 Deliverable | May 11, 2026
# Trainer: Kajola Gbenga Adewale | Prodigy Training Hub
# ═════════════════════════════════════════════════════════════════════════════

# ── CONFIGURATION ────────────────────────────────────────────────────────────
# Change these values to tune the detector for different risk appetites
EQUITY_THRESHOLD = 2.0     # flag equity/balanced/fixed income/dollar > ±2%
MM_THRESHOLD     = 0.5     # flag money market > ±0.5% (tighter rules)
REPORT_LIMIT     = 15      # max anomalies to show in the printed report


# ── HELPER FUNCTIONS ─────────────────────────────────────────────────────────

def get_threshold_for_fund(fund_code):
    """Return the anomaly threshold for a given fund code."""
    return MM_THRESHOLD if fund_code == 'ZMMF' else EQUITY_THRESHOLD


def flag_record_full(record):
    """
    Analyse a fund-day record and return a full flag result.

    Args:
        record (dict): One row from the dataset
    Returns:
        dict: {flagged, type, severity, threshold}
    """
    ret       = record['daily_return_pct']
    threshold = get_threshold_for_fund(record['fund_code'])

    if abs(ret) > threshold:
        flagged   = True
        anom_type = 'LARGE DROP' if ret < 0 else 'LARGE SPIKE'
        # CRITICAL if beyond 2× threshold, else WARNING
        severity  = 'CRITICAL' if abs(ret) > threshold * 2 else 'WARNING'
    else:
        flagged   = False
        anom_type = 'NORMAL'
        severity  = 'NORMAL'

    return {'flagged': flagged, 'type': anom_type, 'severity': severity, 'threshold': threshold}


def format_aum(value_millions):
    """Format AUM as readable string."""
    return f"₦{value_millions:,.1f}M"


# ── MAIN SCAN LOOP ───────────────────────────────────────────────────────────

anomalies_found = []
normal_count    = 0

for record in dataset:
    result = flag_record_full(record)

    if result['flagged']:
        anomalies_found.append({
            'date':        record['date'],
            'fund_code':   record['fund_code'],
            'fund_name':   record['fund_name'],
            'manager':     record['fund_manager'],
            'nav':         record['nav'],
            'return':      record['daily_return_pct'],
            'aum':         record['aum_millions'],
            'type':        result['type'],
            'severity':    result['severity'],
            'threshold':   result['threshold'],
            'true_anomaly': record['is_anomaly']   # ground truth
        })
    else:
        normal_count += 1

print(f"✅ Scan complete — {len(dataset)} records processed.")
print(f"   Anomalies detected: {len(anomalies_found)}")
print(f"   Normal records    : {normal_count}")


✅ Scan complete — 600 records processed.
   Anomalies detected: 28
   Normal records    : 572


In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# ANOMALY REPORT — Summary and full listing
# ─────────────────────────────────────────────────────────────────────────────

print()
print("=" * 80)
print("     ZEDCREST CAPITAL — NAV ANOMALY DETECTION REPORT")
print(f"     Thresholds: Equity > ±{EQUITY_THRESHOLD}%  |  Money Market > ±{MM_THRESHOLD}%")
print("=" * 80)

# ── Grouped by fund ──────────────────────────────────────────────────────────
print("Anomalies by Fund:")
print("-" * 55)
for fc in ['ZMMF', 'ZEGF', 'ZFIF', 'ZDLF', 'ZBAF']:
    fa = [a for a in anomalies_found if a['fund_code'] == fc]
    drops    = sum(1 for a in fa if a['type'] == 'LARGE DROP')
    spikes   = sum(1 for a in fa if a['type'] == 'LARGE SPIKE')
    critical = sum(1 for a in fa if a['severity'] == 'CRITICAL')
    print(
        f"  {fc}  |  Total: {len(fa):>3}  "
        f"|  Drops: {drops:>3}  |  Spikes: {spikes:>3}  |  Critical: {critical:>3}"
    )

# ── Full listing (first REPORT_LIMIT records) ─────────────────────────────────
print()
print(f"Full Anomaly Log (first {REPORT_LIMIT}):")
print("=" * 88)
print(
    f"  {'#':<4}  {'Date':<12}  {'Fund':<6}  {'Return':>8}  "
    f"{'NAV':>10}  {'AUM':>12}  {'Type':<12}  Severity"
)
print("=" * 88)

for i, a in enumerate(anomalies_found[:REPORT_LIMIT], start=1):
    emoji = '🔴' if a['severity'] == 'CRITICAL' else '🟠'
    print(
        f"  {i:<4}  {a['date']:<12}  {a['fund_code']:<6}  "
        f"{a['return']:>+7.2f}%  ₦{a['nav']:>8.4f}  "
        f"{format_aum(a['aum']):>12}  {a['type']:<12}  {emoji} {a['severity']}"
    )

if len(anomalies_found) > REPORT_LIMIT:
    print(f"  ... and {len(anomalies_found) - REPORT_LIMIT} more not shown.")
print("=" * 88)



     ZEDCREST CAPITAL — NAV ANOMALY DETECTION REPORT
     Thresholds: Equity > ±2.0%  |  Money Market > ±0.5%
Anomalies by Fund:
-------------------------------------------------------
  ZMMF  |  Total:   7  |  Drops:   4  |  Spikes:   3  |  Critical:   7
  ZEGF  |  Total:   4  |  Drops:   2  |  Spikes:   2  |  Critical:   0
  ZFIF  |  Total:   6  |  Drops:   3  |  Spikes:   3  |  Critical:   2
  ZDLF  |  Total:   5  |  Drops:   1  |  Spikes:   4  |  Critical:   1
  ZBAF  |  Total:   6  |  Drops:   4  |  Spikes:   2  |  Critical:   1

Full Anomaly Log (first 15):
  #     Date          Fund      Return         NAV           AUM  Type          Severity
  1     2026-01-06    ZMMF      +2.90%  ₦102.9414     ₦4,647.3M  LARGE SPIKE   🔴 CRITICAL
  2     2026-01-13    ZMMF      +2.96%  ₦106.1743     ₦4,829.4M  LARGE SPIKE   🔴 CRITICAL
  3     2026-01-27    ZMMF      -2.42%  ₦104.0942     ₦4,781.8M  LARGE DROP    🔴 CRITICAL
  4     2026-03-13    ZMMF      -3.39%  ₦102.0620     ₦4,970.1M  LARGE

In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# ACCURACY EVALUATION
# We know ground truth because WE planted the anomalies.
# This is how data scientists evaluate detection models.
#
#   True Positives (TP)  : real anomaly, correctly detected
#   False Positives (FP) : normal record, incorrectly flagged
#   False Negatives (FN) : real anomaly, missed by our detector
# ─────────────────────────────────────────────────────────────────────────────

true_positives        = sum(1 for a in anomalies_found if a['true_anomaly'])
false_positives       = sum(1 for a in anomalies_found if not a['true_anomaly'])
total_real_anomalies  = sum(1 for r in dataset if r['is_anomaly'])
false_negatives       = total_real_anomalies - true_positives

# Precision: of everything we flagged, what fraction was genuinely anomalous?
precision = (true_positives / len(anomalies_found) * 100) if anomalies_found else 0

# Recall: of all real anomalies, what fraction did we catch?
recall    = (true_positives / total_real_anomalies * 100) if total_real_anomalies else 0

print()
print("=" * 58)
print("     DETECTOR PERFORMANCE EVALUATION")
print("=" * 58)
print(f"  Real anomalies planted       : {total_real_anomalies}")
print(f"  Anomalies our script flagged : {len(anomalies_found)}")
print()
print(f"  ✅ True Positives  : {true_positives}  (correct alerts)")
print(f"  ❌ False Positives : {false_positives}  (wrong alerts — noise)")
print(f"  ⚠️  False Negatives : {false_negatives}  (real anomalies missed)")
print()
print(f"  Precision : {precision:.1f}%  (of alerts raised, how many were real?)")
print(f"  Recall    : {recall:.1f}%  (of real anomalies, how many did we catch?)")
print("=" * 58)
print()

# Interpret the results
print("Interpretation:")
if recall >= 80:
    print("  ✅ Good recall — catching most real anomalies.")
else:
    print("  ⚠️  Low recall — try LOWERING the threshold to catch more.")

if precision >= 70:
    print("  ✅ Good precision — most alerts are genuine.")
else:
    print("  ⚠️  Low precision — too many false alarms. Try RAISING the threshold.")



     DETECTOR PERFORMANCE EVALUATION
  Real anomalies planted       : 28
  Anomalies our script flagged : 28

  ✅ True Positives  : 28  (correct alerts)
  ❌ False Positives : 0  (wrong alerts — noise)
  ⚠️  False Negatives : 0  (real anomalies missed)

  Precision : 100.0%  (of alerts raised, how many were real?)
  Recall    : 100.0%  (of real anomalies, how many did we catch?)

Interpretation:
  ✅ Good recall — catching most real anomalies.
  ✅ Good precision — most alerts are genuine.


In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# CRITICAL ALERTS — Escalation list for MD and Risk Committee
# ─────────────────────────────────────────────────────────────────────────────

critical_alerts = [a for a in anomalies_found if a['severity'] == 'CRITICAL']

print()
print("=" * 70)
print("     🔴 CRITICAL ALERTS — IMMEDIATE ACTION REQUIRED")
print("=" * 70)

if not critical_alerts:
    print("  ✅ No critical alerts. All funds within acceptable parameters.")
else:
    for a in critical_alerts:
        print()
        print(f"  🔴 CRITICAL — {a['fund_code']} | {a['date']}")
        print(f"     Fund        : {a['fund_name']}")
        print(f"     Manager     : {a['manager']}")
        print(f"     NAV         : ₦{a['nav']:.4f}")
        print(f"     Daily Return: {a['return']:+.2f}%  (Threshold: ±{a['threshold']}%)")
        print(f"     AUM         : {format_aum(a['aum'])}")
        print(f"     Type        : {a['type']}")
        print(f"     Action      :")
        print(f"       1. Contact fund administrator and custodian immediately.")
        print(f"       2. Cross-reference with Bloomberg/Reuters data.")
        print(f"       3. Notify MD and Risk Committee.")
        print(f"       4. File incident report by end of day.")

print()
print("=" * 70)
print("     END OF REPORT — ZEDCREST CAPITAL NAV MONITORING")
print("=" * 70)



     🔴 CRITICAL ALERTS — IMMEDIATE ACTION REQUIRED

  🔴 CRITICAL — ZMMF | 2026-01-06
     Fund        : Zedcrest Money Market Fund
     Manager     : Adebayo Okafor
     NAV         : ₦102.9414
     Daily Return: +2.90%  (Threshold: ±0.5%)
     AUM         : ₦4,647.3M
     Type        : LARGE SPIKE
     Action      :
       1. Contact fund administrator and custodian immediately.
       2. Cross-reference with Bloomberg/Reuters data.
       3. Notify MD and Risk Committee.
       4. File incident report by end of day.

  🔴 CRITICAL — ZMMF | 2026-01-13
     Fund        : Zedcrest Money Market Fund
     Manager     : Adebayo Okafor
     NAV         : ₦106.1743
     Daily Return: +2.96%  (Threshold: ±0.5%)
     AUM         : ₦4,829.4M
     Type        : LARGE SPIKE
     Action      :
       1. Contact fund administrator and custodian immediately.
       2. Cross-reference with Bloomberg/Reuters data.
       3. Notify MD and Risk Committee.
       4. File incident report by end of day.

 

---
## Session 2 — Wrap-Up & Self-Study Assignment

### What We Covered Today

| Topic | Key Concept | Zedcrest Application |
|-------|------------|---------------------|
| Comparison Operators | `==, !=, >, <, >=, <=` | Compare NAV against thresholds |
| Boolean Logic | `and`, `or`, `not` | Multi-condition fund alerts |
| `if / elif / else` | Decision routing | Classify return severity |
| `for` loop | Iterate over collections | Process all 600 records |
| `while` loop | Loop until condition changes | Monitor NAV until breach |
| Functions | Reusable code blocks | `flag_record_full()`, `classify_return()` |
| Capstone | Full pipeline | Production-ready anomaly detector |

---

### Self-Study Assignment (Complete Before Friday, May 15)

**Task 1 — Extend the Anomaly Script:**  
Add a function called `generate_fund_summary(dataset, fund_code)` that returns a dictionary containing:  
- `total_anomalies`, `worst_day`, `best_day`, `avg_return`, `total_return_pct`  
Call it for all 5 funds and print a formatted summary table.

**Task 2 — New Business Rule:**  
Flag any day where a fund's return is more than **3× its benchmark return** (in either direction), even if the absolute return is within the normal threshold.  
Modify `flag_record_full()` to include this rule using the `benchmark_return_pct` column.

**Task 3 — Thinking Question (answer in a comment):**  
- What happens to precision and recall if you LOWER the threshold from 2.0% to 1.0%?  
- What if you RAISE it to 3.0%?  
- For Zedcrest, which is more dangerous: a false positive or a false negative? Why?

---

### Session 3 Preview (Friday, May 15)

**Topic:** Data Structures — Lists, Tuples, Dictionaries, Sets & List Comprehensions  
**Applied to:** Fund registries, client ID lookups, date range filtering, portfolio dictionaries  
**Deliverable:** A fund registry system using dictionaries and a client deduplication tool using sets

---

> *'The for loop is not just a coding tool — it is the mechanism by which computers process in one second what would take a human analyst a week. Master it, and you have fundamentally changed what you are capable of producing.'*  
> — Kajola Gbenga, Session 2

---
**ZEDCREST CAPITAL | Data Analyst Training Programme | May – August 2026**  
*CONFIDENTIAL — Training Provider: Kajola Gbenga Adewale / Prodigy Training Hub*
